In [5]:
#!/usr/bin/env python3
"""
F1 2026 Japanese Grand Prix Predictor — FastF1 Edition
======================================================
Uses FastF1 API to pull real session data (results, lap times, telemetry)
then trains a Gradient Boosting + Random Forest ensemble to predict Round 3.

Requirements:
    pip install fastf1 scikit-learn pandas numpy

Usage:
    python f1_fastf1_predict.py

The script will:
  1. Load all 2026 Race & Sprint sessions via FastF1
  2. Load 2024-2025 Japanese GP sessions for track-specific history
  3. Engineer features from real timing data
  4. Train an ensemble ML model
  5. Output predicted finishing order for the Japanese GP
  6. Export predictions.json for the React dashboard
"""

import json
import warnings
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import fastf1

from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import cross_val_score

warnings.filterwarnings("ignore")

# ─── CONFIG ───────────────────────────────────────────────────────────
CURRENT_YEAR = 2026
PREDICT_ROUND = 3          # Japanese GP
HISTORY_YEARS = [2024, 2025]  # Historical seasons to include
COMPLETED_ROUNDS = 2        # How many 2026 rounds have been completed
CACHE_DIR = Path.home() / ".cache" / "fastf1"

# Enable FastF1 cache for faster reloads
# 定义缓存路径
CACHE_DIR = Path.home() / ".cache" / "fastf1"

# ✨ 核心修复：如果文件夹不存在，就递归创建它
if not CACHE_DIR.exists():
    CACHE_DIR.mkdir(parents=True, exist_ok=True)
    print(f"已为你创建缓存目录: {CACHE_DIR}")

# 现在再启用缓存就不会报错了
fastf1.Cache.enable_cache(str(CACHE_DIR))

# ─── 1. DATA LOADING via FastF1 ──────────────────────────────────────

def load_2026_sessions() -> dict[int, pd.DataFrame]:
    """
    Load all completed 2026 Race + Sprint sessions.
    Follows the user's tested pattern.
    """
    all_sessions = {}
    session_idx = 0

    for rnd in range(1, COMPLETED_ROUNDS + 1):
        # ── Main Race ──
        print(f"  Loading 2026 Round {rnd} Race...")
        try:
            session = fastf1.get_session(CURRENT_YEAR, rnd, 'Race')
            session.load()
            results = session.results.copy()
            results["SessionType"] = "Race"
            results["Round"] = rnd
            results["Year"] = CURRENT_YEAR
            results["EventName"] = session.event["EventName"]
            all_sessions[session_idx] = {
                "results": results,
                "laps": session.laps.copy() if hasattr(session, 'laps') and session.laps is not None else None,
                "type": "Race",
                "round": rnd,
                "event": session.event["EventName"],
            }
            session_idx += 1
        except Exception as e:
            print(f"    ⚠ Failed to load Race R{rnd}: {e}")

        # ── Sprint (not every round has one) ──
        try:
            print(f"  Loading 2026 Round {rnd} Sprint...")
            session_s = fastf1.get_session(CURRENT_YEAR, rnd, 'S')
            session_s.load()
            results_s = session_s.results.copy()
            results_s["SessionType"] = "Sprint"
            results_s["Round"] = rnd
            results_s["Year"] = CURRENT_YEAR
            results_s["EventName"] = session_s.event["EventName"]
            all_sessions[session_idx] = {
                "results": results_s,
                "laps": session_s.laps.copy() if hasattr(session_s, 'laps') and session_s.laps is not None else None,
                "type": "Sprint",
                "round": rnd,
                "event": session_s.event["EventName"],
            }
            session_idx += 1
        except Exception as e:
            print(f"    ⚠ Round {rnd} 没有 Sprint，跳过 ({e})")

    print(f"  ✅ Loaded {session_idx} sessions from 2026\n")
    return all_sessions


def load_historical_japan() -> dict[int, pd.DataFrame]:
    """Load Japanese GP Race results from previous seasons."""
    japan_sessions = {}
    for year in HISTORY_YEARS:
        print(f"  Loading {year} Japanese GP...")
        try:
            session = fastf1.get_session(year, 'Japan', 'Race')
            session.load()
            results = session.results.copy()
            results["Year"] = year
            results["SessionType"] = "Race"
            results["EventName"] = "Japanese Grand Prix"
            japan_sessions[year] = {
                "results": results,
                "laps": session.laps.copy() if hasattr(session, 'laps') and session.laps is not None else None,
            }
        except Exception as e:
            print(f"    ⚠ Failed to load {year} Japan: {e}")
    print(f"  ✅ Loaded {len(japan_sessions)} historical Japan sessions\n")
    return japan_sessions


# ─── 2. FEATURE ENGINEERING from FastF1 data ─────────────────────────

# 2026 driver → team mapping (will be auto-populated from FastF1 data)
DRIVER_TEAM = {}

# Track characteristics: Suzuka
SUZUKA = {
    "power_sensitivity": 0.85,
    "aero_sensitivity": 0.90,
    "tire_deg": 0.70,
    "overtaking": 0.75,
}

# Power Unit manufacturer groupings
PU_MANUFACTURER = {
    "Mercedes": "Mercedes",
    "Ferrari": "Ferrari",
    "McLaren": "Mercedes",     # Customer PU
    "Red Bull Racing": "Ford", "Red Bull": "Ford",
    "Haas F1 Team": "Ferrari", "Haas": "Ferrari",
    "Alpine": "Mercedes",      # Switched from Renault for 2026
    "RB": "Ford", "Racing Bulls": "Ford",
    "Williams": "Mercedes",
    "Audi": "Audi",  # New works PU
    "Kick Sauber": "Audi",
    "Aston Martin": "Honda",   # New Honda partnership
    "Cadillac": "Ferrari",     # Customer PU
}


def extract_driver_stats(all_sessions_2026: dict) -> pd.DataFrame:
    """
    Extract per-driver statistics from 2026 FastF1 session results.
    Returns a DataFrame indexed by driver abbreviation.
    """
    records = []

    for idx, sess in all_sessions_2026.items():
        res = sess["results"]
        laps = sess["laps"]
        sess_type = sess["type"]
        rnd = sess["round"]

        for _, row in res.iterrows():
            abbr = row.get("Abbreviation", "")
            if not abbr or pd.isna(abbr):
                continue

            # Map driver → team
            team_name = row.get("TeamName", "Unknown")
            full_name = row.get("FullName", row.get("BroadcastName", abbr))
            DRIVER_TEAM[abbr] = {"team": team_name, "full_name": full_name}

            # Position (handle DNF/DNS)
            pos = row.get("Position", None)
            if pd.isna(pos):
                pos = 22  # DNS/DNF → worst
            else:
                pos = int(pos)

            grid_pos = row.get("GridPosition", None)
            if pd.isna(grid_pos):
                grid_pos = 20
            else:
                grid_pos = int(grid_pos)

            status = str(row.get("Status", ""))
            finished = "Finished" in status or "+" in status or status == ""
            dnf = not finished and pos > 0

            # Points scored
            pts = row.get("Points", 0)
            if pd.isna(pts):
                pts = 0

            # Lap time stats (from laps data if available)
            avg_lap_seconds = None
            best_lap_seconds = None
            consistency_std = None

            if laps is not None and len(laps) > 0:
                driver_laps = laps[laps["Driver"] == abbr]
                if len(driver_laps) > 0:
                    # Filter valid lap times (exclude pit laps, SC laps, etc.)
                    valid_laps = driver_laps.dropna(subset=["LapTime"])
                    if len(valid_laps) > 0:
                        lap_seconds = valid_laps["LapTime"].dt.total_seconds()
                        # Remove obvious outliers (pit laps, SC) — keep within 110% of best
                        best = lap_seconds.min()
                        clean_laps = lap_seconds[lap_seconds < best * 1.10]
                        if len(clean_laps) > 0:
                            avg_lap_seconds = clean_laps.mean()
                            best_lap_seconds = clean_laps.min()
                            consistency_std = clean_laps.std()

            records.append({
                "Abbreviation": abbr,
                "FullName": full_name,
                "TeamName": team_name,
                "Round": rnd,
                "SessionType": sess_type,
                "Position": pos,
                "GridPosition": grid_pos,
                "Finished": finished,
                "DNF": dnf,
                "Points": float(pts),
                "AvgLapSec": avg_lap_seconds,
                "BestLapSec": best_lap_seconds,
                "ConsistencyStd": consistency_std,
            })

    return pd.DataFrame(records)


def extract_japan_history(japan_sessions: dict) -> pd.DataFrame:
    """Extract driver performance at Suzuka from historical data."""
    records = []
    for year, sess in japan_sessions.items():
        res = sess["results"]
        for _, row in res.iterrows():
            abbr = row.get("Abbreviation", "")
            if not abbr or pd.isna(abbr):
                continue
            pos = row.get("Position", 22)
            if pd.isna(pos):
                pos = 22
            grid_pos = row.get("GridPosition", 20)
            if pd.isna(grid_pos):
                grid_pos = 20
            records.append({
                "Abbreviation": abbr,
                "Year": year,
                "JapanPosition": int(pos),
                "JapanGrid": int(grid_pos),
            })
    return pd.DataFrame(records)


def build_driver_features(stats_2026: pd.DataFrame, japan_hist: pd.DataFrame) -> pd.DataFrame:
    """
    Aggregate per-driver features for prediction.
    Returns one row per driver with engineered features.
    """
    drivers = stats_2026["Abbreviation"].unique()
    features_list = []

    for abbr in drivers:
        d = stats_2026[stats_2026["Abbreviation"] == abbr]
        info = DRIVER_TEAM.get(abbr, {"team": "Unknown", "full_name": abbr})
        team = info["team"]

        # ── Recent form ──
        race_results = d[d["SessionType"] == "Race"]
        sprint_results = d[d["SessionType"] == "Sprint"]
        all_results = d

        avg_finish = all_results["Position"].mean()
        best_finish = all_results["Position"].min()
        worst_finish = all_results["Position"].max()
        finish_std = all_results["Position"].std() if len(all_results) > 1 else 5.0

        avg_grid = all_results["GridPosition"].mean()
        best_grid = all_results["GridPosition"].min()

        # Positions gained/lost (grid → finish)
        all_results_copy = all_results.copy()
        all_results_copy["PosGain"] = all_results_copy["GridPosition"] - all_results_copy["Position"]
        avg_pos_gain = all_results_copy["PosGain"].mean()

        # Reliability
        total_entries = len(race_results)
        dnf_count = race_results["DNF"].sum()
        dns_count = (race_results["Position"] >= 22).sum()  # approx DNS
        reliability = 1.0 - (dnf_count / max(total_entries, 1))

        # Points per race
        total_points = all_results["Points"].sum()
        points_per_race = total_points / max(len(race_results), 1)

        # Pace metrics (from lap times)
        avg_lap = all_results["AvgLapSec"].dropna().mean() if all_results["AvgLapSec"].notna().any() else None
        best_lap = all_results["BestLapSec"].dropna().min() if all_results["BestLapSec"].notna().any() else None
        consistency = all_results["ConsistencyStd"].dropna().mean() if all_results["ConsistencyStd"].notna().any() else None

        # ── Japan-specific history ──
        jh = japan_hist[japan_hist["Abbreviation"] == abbr] if len(japan_hist) > 0 else pd.DataFrame()
        japan_avg_pos = jh["JapanPosition"].mean() if len(jh) > 0 else 12.0
        japan_avg_grid = jh["JapanGrid"].mean() if len(jh) > 0 else 12.0
        japan_experience = len(jh)

        # ── Team / PU factors ──
        pu = PU_MANUFACTURER.get(team, "Unknown")
        pu_score = {"Mercedes": 5, "Ferrari": 4, "Honda": 3, "Ford": 2, "Audi": 1}.get(pu, 1)
        # Suzuka power bonus
        pu_bonus = SUZUKA["power_sensitivity"] * pu_score
        # Honda home race bonus at Suzuka
        home_bonus = 2.0 if pu == "Honda" else 0.0

        features_list.append({
            "Abbreviation": abbr,
            "FullName": info["full_name"],
            "TeamName": team,
            # Core features
            "avg_finish": avg_finish,
            "best_finish": best_finish,
            "worst_finish": worst_finish,
            "finish_std": finish_std,
            "avg_grid": avg_grid,
            "best_grid": best_grid,
            "avg_pos_gain": avg_pos_gain,
            "reliability": reliability,
            "points_per_race": points_per_race,
            "total_points": total_points,
            # Pace
            "avg_lap_sec": avg_lap if avg_lap else 0,
            "best_lap_sec": best_lap if best_lap else 0,
            "lap_consistency": consistency if consistency else 3.0,
            # Japan-specific
            "japan_avg_pos": japan_avg_pos,
            "japan_avg_grid": japan_avg_grid,
            "japan_experience": japan_experience,
            # Track/PU
            "pu_bonus": pu_bonus,
            "home_bonus": home_bonus,
            # Entries count (for weighting)
            "num_entries": len(all_results),
        })

    return pd.DataFrame(features_list)


# ─── 3. MODEL TRAINING & PREDICTION ──────────────────────────────────

FEATURE_COLS = [
    "avg_finish", "best_finish", "worst_finish", "finish_std",
    "avg_grid", "best_grid", "avg_pos_gain",
    "reliability", "points_per_race",
    "avg_lap_sec", "best_lap_sec", "lap_consistency",
    "japan_avg_pos", "japan_avg_grid", "japan_experience",
    "pu_bonus", "home_bonus",
]


def build_training_data(stats_2026: pd.DataFrame, driver_features: pd.DataFrame) -> tuple:
    """
    Build X, y training arrays.
    Each sample = one driver's features → their finishing position in a session.
    We augment by creating one training sample per session appearance.
    """
    X_rows = []
    y_rows = []

    for _, row in stats_2026.iterrows():
        abbr = row["Abbreviation"]
        feat_row = driver_features[driver_features["Abbreviation"] == abbr]
        if len(feat_row) == 0:
            continue
        feat = feat_row.iloc[0]

        x = [feat[col] for col in FEATURE_COLS]
        y = row["Position"]

        # Weight races more than sprints
        weight = 2 if row["SessionType"] == "Race" else 1
        for _ in range(weight):
            X_rows.append(x)
            y_rows.append(min(y, 20))

    return np.array(X_rows, dtype=float), np.array(y_rows, dtype=float)


def train_and_predict(X_train, y_train, driver_features: pd.DataFrame) -> pd.DataFrame:
    """Train ensemble and predict Japanese GP finishing order."""

    print("Training models...")

    # Gradient Boosting
    gb = GradientBoostingRegressor(
        n_estimators=200, max_depth=4, learning_rate=0.1,
        min_samples_split=3, random_state=42,
    )
    gb.fit(X_train, y_train)

    # Random Forest
    rf = RandomForestRegressor(
        n_estimators=200, max_depth=5, min_samples_split=3, random_state=42,
    )
    rf.fit(X_train, y_train)

    # Cross-validation
    gb_cv = cross_val_score(gb, X_train, y_train, cv=min(3, len(X_train)), scoring="neg_mean_absolute_error")
    rf_cv = cross_val_score(rf, X_train, y_train, cv=min(3, len(X_train)), scoring="neg_mean_absolute_error")

    print(f"  Gradient Boosting MAE: {-gb_cv.mean():.2f} positions")
    print(f"  Random Forest MAE:     {-rf_cv.mean():.2f} positions")

    # Feature importances
    print(f"\n  Top 5 Feature Importances (Gradient Boosting):")
    importances = gb.feature_importances_
    sorted_idx = np.argsort(importances)[::-1]
    for i in sorted_idx[:5]:
        print(f"    {FEATURE_COLS[i]:<20s} {importances[i]:.3f}")

    # Predict for all drivers
    predictions = []
    for _, feat in driver_features.iterrows():
        x = np.array([[feat[col] for col in FEATURE_COLS]], dtype=float)
        gb_pred = gb.predict(x)[0]
        rf_pred = rf.predict(x)[0]
        ensemble = 0.6 * gb_pred + 0.4 * rf_pred

        spread = abs(gb_pred - rf_pred)
        if spread < 1:
            conf, conf_pct = "HIGH", min(95, int(90 - spread * 10))
        elif spread < 2.5:
            conf, conf_pct = "MED", min(80, int(75 - spread * 5))
        else:
            conf, conf_pct = "LOW", max(40, int(60 - spread * 5))

        predictions.append({
            "Abbreviation": feat["Abbreviation"],
            "FullName": feat["FullName"],
            "TeamName": feat["TeamName"],
            "score": ensemble,
            "gb_pred": gb_pred,
            "rf_pred": rf_pred,
            "confidence": conf,
            "confidence_pct": conf_pct,
            "total_points": feat["total_points"],
        })

    pred_df = pd.DataFrame(predictions).sort_values("score").reset_index(drop=True)
    pred_df["predicted_position"] = range(1, len(pred_df) + 1)

    return pred_df, {
        "gb_mae": round(-gb_cv.mean(), 2),
        "rf_mae": round(-rf_cv.mean(), 2),
        "training_samples": len(X_train),
        "features": len(FEATURE_COLS),
        "feature_importances": {
            FEATURE_COLS[i]: round(float(importances[i]), 4)
            for i in sorted_idx
        },
    }


# ─── 4. OUTPUT ────────────────────────────────────────────────────────

POINTS_SYSTEM = {1: 25, 2: 18, 3: 15, 4: 12, 5: 10, 6: 8, 7: 6, 8: 4, 9: 2, 10: 1}


def print_results(pred_df: pd.DataFrame, model_info: dict):
    """Pretty-print the predicted grid."""
    print(f"\n{'=' * 65}")
    print(f"  PREDICTED RACE RESULT — 2026 JAPANESE GRAND PRIX (SUZUKA)")
    print(f"{'=' * 65}")
    print(f"{'Pos':<5}{'Driver':<25}{'Team':<18}{'Score':<8}{'Conf'}")
    print(f"{'-' * 65}")

    for _, row in pred_df.iterrows():
        pos = row["predicted_position"]
        pts = POINTS_SYSTEM.get(pos, 0)
        pts_str = f"+{pts}pts" if pts > 0 else ""
        print(
            f"P{pos:<4}"
            f"{row['FullName']:<25}"
            f"{row['TeamName']:<18}"
            f"{row['score']:<8.2f}"
            f"{row['confidence']} ({row['confidence_pct']}%) {pts_str}"
        )

    print(f"\n{'=' * 65}")
    print(f"  MODEL INFO")
    print(f"{'=' * 65}")
    print(f"  Training samples: {model_info['training_samples']}")
    print(f"  Features:         {model_info['features']}")
    print(f"  GB MAE:           {model_info['gb_mae']} positions")
    print(f"  RF MAE:           {model_info['rf_mae']} positions")


def export_json(pred_df: pd.DataFrame, model_info: dict, path: str = "predictions.json"):
    """Export predictions as JSON for the React dashboard."""
    preds = []
    for _, row in pred_df.iterrows():
        pos = row["predicted_position"]
        preds.append({
            "position": int(pos),
            "driver": row["FullName"],
            "abbreviation": row["Abbreviation"],
            "team": row["TeamName"],
            "score": round(row["score"], 2),
            "confidence": row["confidence"],
            "confidence_pct": int(row["confidence_pct"]),
            "gb_pred": round(row["gb_pred"], 2),
            "rf_pred": round(row["rf_pred"], 2),
            "current_points": round(row["total_points"], 1),
        })

    output = {
        "race": "Japanese Grand Prix 2026",
        "circuit": "Suzuka Circuit",
        "round": PREDICT_ROUND,
        "date": "March 27-29, 2026",
        "predictions": preds,
        "model_info": model_info,
    }

    with open(path, "w") as f:
        json.dump(output, f, indent=2)
    print(f"\n  ✅ Exported to {path}")


# ─── MAIN ─────────────────────────────────────────────────────────────

def main():
    print("=" * 65)
    print("  F1 2026 JAPANESE GP PREDICTOR — FastF1 Edition")
    print("  Powered by: fastf1 + scikit-learn")
    print("=" * 65)

    # 1. Load data
    print("\n[1/5] Loading 2026 sessions from FastF1...")
    sessions_2026 = load_2026_sessions()

    print("[2/5] Loading historical Japanese GP data...")
    japan_sessions = load_historical_japan()

    # 2. Extract stats
    print("[3/5] Extracting features from session data...")
    stats_2026 = extract_driver_stats(sessions_2026)
    japan_hist = extract_japan_history(japan_sessions)
    driver_features = build_driver_features(stats_2026, japan_hist)

    print(f"  Drivers found: {len(driver_features)}")
    print(f"  Total session entries: {len(stats_2026)}")
    print(f"  Japan history records: {len(japan_hist)}")

    # 3. Build training data
    print("\n[4/5] Building training data & training models...")
    X_train, y_train = build_training_data(stats_2026, driver_features)
    print(f"  Training samples: {len(X_train)} (weighted)")

    # Handle NaN/inf
    X_train = np.nan_to_num(X_train, nan=0.0, posinf=0.0, neginf=0.0)

    # 4. Train & predict
    pred_df, model_info = train_and_predict(X_train, y_train, driver_features)

    # 5. Output
    print("\n[5/5] Results:")
    print_results(pred_df, model_info)
    export_json(pred_df, model_info)

    print("\n🏁 Done! Run the React dashboard to visualize predictions.")


if __name__ == "__main__":
    main()

已为你创建缓存目录: /Users/junyuyao/.cache/fastf1
  F1 2026 JAPANESE GP PREDICTOR — FastF1 Edition
  Powered by: fastf1 + scikit-learn

[1/5] Loading 2026 sessions from FastF1...
  Loading 2026 Round 1 Race...


core           INFO 	Loading data for Australian Grand Prix - Race [v3.7.0]
req            INFO 	No cached data found for session_info. Loading data...
_api           INFO 	Fetching session info data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for driver_info. Loading data...
_api           INFO 	Fetching driver list...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for session_status_data. Loading data...
_api           INFO 	Fetching session status data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for lap_count. Loading data...
_api           INFO 	Fetching lap count data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for track_status_data. Loading data...
_api           INFO 	Fetching track status data...
req            INFO 	Data has been written to cache!
req            INFO 	No

  Loading 2026 Round 1 Sprint...
    ⚠ Round 1 没有 Sprint，跳过 (Session type 'S' does not exist for this event)
  Loading 2026 Round 2 Race...


req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for driver_info. Loading data...
_api           INFO 	Fetching driver list...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for session_status_data. Loading data...
_api           INFO 	Fetching session status data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for lap_count. Loading data...
_api           INFO 	Fetching lap count data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for track_status_data. Loading data...
_api           INFO 	Fetching track status data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for _extended_timing_data. Loading data...
_api           INFO 	Fetching timing data...
_api           INFO 	Parsing timing data...
req            INFO 	Data has been written to cache!

  Loading 2026 Round 2 Sprint...


...

  ✅ Loaded 3 sessions from 2026

[2/5] Loading historical Japanese GP data...
  Loading 2024 Japanese GP...


core           INFO 	Loading data for Japanese Grand Prix - Race [v3.7.0]
req            INFO 	No cached data found for session_info. Loading data...
_api           INFO 	Fetching session info data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for driver_info. Loading data...
_api           INFO 	Fetching driver list...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for session_status_data. Loading data...
_api           INFO 	Fetching session status data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for lap_count. Loading data...
_api           INFO 	Fetching lap count data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for track_status_data. Loading data...
_api           INFO 	Fetching track status data...
req            INFO 	Data has been written to cache!
req            INFO 	No c

  Loading 2025 Japanese GP...


core           INFO 	Loading data for Japanese Grand Prix - Race [v3.7.0]
req            INFO 	No cached data found for session_info. Loading data...
_api           INFO 	Fetching session info data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for driver_info. Loading data...
_api           INFO 	Fetching driver list...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for session_status_data. Loading data...
_api           INFO 	Fetching session status data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for lap_count. Loading data...
_api           INFO 	Fetching lap count data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for track_status_data. Loading data...
_api           INFO 	Fetching track status data...
req            INFO 	Data has been written to cache!
req            INFO 	No c

  ✅ Loaded 2 historical Japan sessions

[3/5] Extracting features from session data...
  Drivers found: 22
  Total session entries: 66
  Japan history records: 40

[4/5] Building training data & training models...
  Training samples: 110 (weighted)
Training models...
  Gradient Boosting MAE: 3.80 positions
  Random Forest MAE:     3.71 positions

  Top 5 Feature Importances (Gradient Boosting):
    worst_finish         0.737
    avg_finish           0.180
    reliability          0.033
    japan_avg_grid       0.012
    lap_consistency      0.011

[5/5] Results:

  PREDICTED RACE RESULT — 2026 JAPANESE GRAND PRIX (SUZUKA)
Pos  Driver                   Team              Score   Conf
-----------------------------------------------------------------
P1   George Russell           Mercedes          1.40    HIGH (89%) +25pts
P2   Kimi Antonelli           Mercedes          2.19    HIGH (89%) +18pts
P3   Charles Leclerc          Ferrari           3.19    HIGH (89%) +15pts
P4   Lewis Hamilton  

du -sh ~/.cache/fastf1

To Clean up your cache